# Future Property Price Prediction

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from dotenv import load_dotenv
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
import mlflow
from mlflow.data import from_pandas
import mlflow.pyfunc

In [ ]:
data = pd.read_csv('Datasets/Future_Price.csv')

In [ ]:
data.info()

### Domain Data Validation

In [ ]:
data.describe(include='all')

In [ ]:
data = data[data['Floor_No'] <= data['Total_Floors']]

### Data Preprocessing

In [ ]:
# Converting categorical to numerical
binary_map = {
  'Yes':1,
  'No':0
}

cols = ['Parking_Space', 'Security']
for col in cols:
  data[col] = data[col].map(binary_map)

### Feature Correlation with Target Variable

In [ ]:
corr = data.corr(numeric_only=True)
plt.figure(figsize=(20,12))
sns.heatmap(
  data=corr,
  annot=True,
  cmap='YlGnBu'
)
plt.show()

### Feature Variable and Target variable

In [ ]:
X = data.drop(columns=['ID', 'Future_Price_5Y', 'Price_in_Lakhs', 'Amenity_Score', 'Year_Built'])
y = data['Future_Price_5Y']

### Feature selection using Mutual Information

In [ ]:
"""X_encoded = X.copy()

for col in X_encoded.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X_encoded[col])

mi_scores = mutual_info_regression(X_encoded, y)

mi_df = pd.DataFrame({
    'Feature': X_encoded.columns,
    'MI_Score': mi_scores
})

mi_df = mi_df.sort_values(by='MI_Score', ascending=False)

print(mi_df)"""

In [ ]:
"""mi_df.sort_values(by='MI_Score').plot(
    x='Feature',
    y='MI_Score',
    kind='barh',
    figsize=(8,6)
)

plt.title("Feature Importance (Information Gain)")
plt.show()"""

# Linear Regression

### Data Preprocessing Pipeline

In [ ]:
# Target Encoding City and Locality
class LocationFeatureEngineer(BaseEstimator, TransformerMixin):
  def __init__(self, city_col='City', locality_col='Locality'):
    self.city_col = city_col
    self.locality_col = locality_col

  def extract_locality_number(self, val):
    val_list = val.split("_")
    return str(val_list[1])

  def fit(self, X, y=None):
    return self

  def transform(self, X):
    X = X.copy()

    # Extract locality number
    X[self.locality_col] = X[self.locality_col].apply(self.extract_locality_number)

    # Create Location feature
    X['Location'] = X[self.city_col] + '_' + X[self.locality_col]

    # Drop original columns
    X.drop(columns=[self.city_col, self.locality_col], inplace=True)

    return X


class SmoothedTargetEncoder(BaseEstimator, TransformerMixin):
  def __init__(self, col, smoothing=10):
    self.col = col
    self.smoothing = smoothing

  def fit(self, X, y):
    df = pd.DataFrame({
      self.col: X[self.col],
      'target': y
    })

    stats = df.groupby(self.col)['target'].agg(['mean', 'count'])

    self.global_mean_ = y.mean()

    # Smoothing formula
    self.mapping_ = (
      (stats['mean'] * stats['count'] + self.global_mean_ * self.smoothing) /
      (stats['count'] + self.smoothing)
    )

    return self

  def transform(self, X):
    X = X.copy()

    X[self.col] = X[self.col].map(self.mapping_)

    # Handle unseen values
    X[self.col] = X[self.col].fillna(self.global_mean_)

    return X

location_pipeline = Pipeline([
  ('location_feature_engineering', LocationFeatureEngineer()),
  ('target_enc', SmoothedTargetEncoder(col='Location', smoothing=10))
])

cat_pipeline = Pipeline([
  ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

std_sle_pipeline = Pipeline([
  ('scaler', StandardScaler())
])

ordinal_pipeline = Pipeline([
  ('ordinal', OrdinalEncoder())
])

binary_pipeline = 'passthrough'


### Final Model Pipeline

In [ ]:
# selected_features = mi_df[mi_df['MI_Score'] > 0]['Feature'].tolist()
selected_features = X.columns

lr_tar_var = []
lr_num_var = []
lr_cat_var = []
lr_ord_var = []
lr_bin_var = []

for var in selected_features:
    
    unique_vals = data[var].dropna().unique()

    
    if set(unique_vals).issubset({0, 1}):
        lr_bin_var.append(var)

    
    elif (data[var].dtype == 'float') or (data[var].dtype == 'int'):
        lr_num_var.append(var)
    
    elif data[var].dtype == 'object':
        if var in ['City', 'Locality']:
            lr_tar_var.append(var)
        elif var == 'Public_Transport_Accessibility':
            lr_ord_var.append(var)
        else:
            lr_cat_var.append(var)

lr_preprocessor = ColumnTransformer([
    ('location', location_pipeline, lr_tar_var),
    ('cat', cat_pipeline, lr_cat_var),
    ('num', std_sle_pipeline, lr_num_var),
    ('ordinal', ordinal_pipeline, lr_ord_var),
    ('bin', binary_pipeline, lr_bin_var)
])

lr_pipeline = Pipeline([
    ('preprocessing', lr_preprocessor),
    ('model', LinearRegression())
])

lr_tags = {
  "Feature Selection": "mutual_info_regression",
  "MI": "> 0",
  "model": "Linear Regression",
  "NumericalScalar":"StandardScaler"
}

print(lr_tar_var)
print(lr_num_var)
print(lr_cat_var)
print(lr_ord_var)
print(lr_bin_var)

# Ridge Regression

In [ ]:
ridge_pipeline = Pipeline([
    ('preprocessing', lr_preprocessor), # Same preprocessing as Linear Regression
    ('model', Ridge(
        alpha=1.0         # Regularization strength (tune this)
    ))
])

param_grid = {
    'model__alpha': [1e-4, 1e-3, 1e-2, 0.1, 1, 10, 100, 1000]
}

ridge_grid = GridSearchCV(
    ridge_pipeline,
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

ridge_tags = {
  "Feature Selection": "Auto",
  "model": "Ridge Regression",
  "Regularization": "L2",
  "NumericalScalar": "StandardScaler"
}

# Lasso Regression

In [ ]:
lasso_pipeline = Pipeline([
  ('preprocessing', lr_preprocessor),  # reuse same preprocessing
  ('model', Lasso(
      max_iter=10000,   # important for convergence
      random_state=42
  ))
])

param_grid = {
    'model__alpha': [1e-4, 1e-3, 1e-2, 0.1, 1, 10]
}

lasso_grid = GridSearchCV(
  lasso_pipeline,
  param_grid,
  cv=5,
  scoring='r2',
  n_jobs=-1
)

lasso_tags = {
  "Feature Selection": "L1 Regularization (Auto)",
  "model": "Lasso Regression",
  "Regularization": "L1",
  "NumericalScalar": "StandardScaler"
}

# Tree-Based Regression Models

### Feature Selection

In [ ]:
selected_features = X.columns

rf_tar_var = []
rf_num_var = []
rf_cat_var = []
rf_ord_var = []
rf_bin_var = []

for var in selected_features:

    # Drop NA for safety when checking unique values
    unique_vals = data[var].dropna().unique()

    # ✅ Step 1: Detect Binary (0/1)
    if set(unique_vals).issubset({0, 1}):
        rf_bin_var.append(var)

    # ✅ Step 2: Numerical / Ordinal
    elif (data[var].dtype == 'float') or (data[var].dtype == 'int'):
        rf_num_var.append(var)

    # ✅ Step 3: Categorical
    elif data[var].dtype == 'object':
        if var in ['City', 'Locality']:
            rf_tar_var.append(var)
        elif var == 'Public_Transport_Accessibility':
            rf_ord_var.append(var)
        else:
            rf_cat_var.append(var)

print(rf_tar_var)
print(rf_num_var)
print(rf_cat_var)
print(rf_ord_var)
print(rf_bin_var)

### Data Preprocessing Pipeline

In [ ]:
class TargetEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cols=None):
        self.cols = cols

    def fit(self, X, y):
        X = pd.DataFrame(X, columns=self.cols)
        self.global_mean = y.mean()
        self.target_means = {}

        for col in self.cols:
            self.target_means[col] = y.groupby(X[col]).mean()

        return self

    def transform(self, X):
        X = pd.DataFrame(X, columns=self.cols)

        for col in self.cols:
            X[col] = X[col].map(self.target_means[col])
            X[col] = X[col].fillna(self.global_mean)

        return X.values

class OrdinalMapper(BaseEstimator, TransformerMixin):
    def __init__(self, mapping):
        self.mapping = mapping

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = pd.DataFrame(X, columns=self.mapping.keys())

        for col, map_dict in self.mapping.items():
            X[col] = X[col].map(map_dict)

        return X.values


# ----------------------------------
# Ordinal Mapping
# ----------------------------------
ordinal_mapping = {
    'Public_Transport_Accessibility': {
        'Low': 0,
        'Medium': 1,
        'High': 2
    }
}

# ----------------------------------
# Column Transformer
# ----------------------------------
tree_preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', rf_num_var),
        ('bin', 'passthrough', rf_bin_var),

        ('low_cat',
         OneHotEncoder(handle_unknown='ignore'),
         rf_cat_var),

        ('high_cat',
         TargetEncoder(cols=rf_tar_var),
         rf_tar_var),

        ('ordinal',
         OrdinalMapper(mapping=ordinal_mapping),
         rf_ord_var)
    ]
)


## Decision Tree Regressor

In [ ]:
dt_pipeline = Pipeline([
    ('preprocessing', tree_preprocessor), 
    ('model', DecisionTreeRegressor(
        max_depth=15,
        min_samples_leaf=2,
        random_state=42
    ))
])

dt_tags = {
  "model": "Decision Tree Regressor",
  "Tree Type": "Single Tree",
  "NumericalScalar": "Not Required",
  "Handles Non-linearity": "Yes"
}

## Random Forest Regressor

In [ ]:
rf_pipeline = Pipeline([
    ('preprocessing', tree_preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=100,
        max_depth=15,
        random_state=42,
        min_samples_leaf=2,
        n_jobs=-1
    ))
])

rf_tags = {
  "model": "Random Forest Regressor",
  "NumericalScalar":"StandardScaler"
}

# XGBoost Regressor

In [ ]:
xgb_pipeline = Pipeline([
    ('preprocessing', tree_preprocessor),
    ('model', XGBRegressor(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    ))
])

xgb_tags = {
    "model": "XGBoost Regressor",
    "Tree Type": "Boosted Trees",
    "NumericalScalar": "Not Required",
    "Handles Non-linearity": "Yes",
    "Handles Missing Values": "Yes"
}

# Model Performance Visualization

In [ ]:
def log_regression_plot(y_true, y_pred, model_name, dataset_type="test"):

    plt.figure()

    # Scatter plot
    plt.scatter(y_true, y_pred, alpha=0.5)

    # Ideal line (perfect prediction)
    min_val = min(min(y_true), min(y_pred))
    max_val = max(max(y_true), max(y_pred))
    plt.plot([min_val, max_val], [min_val, max_val])

    plt.xlabel("Actual Values")
    plt.ylabel("Predicted Values")
    plt.title(f"{model_name} - {dataset_type} (Actual vs Predicted)")

    # Save plot
    os.makedirs("Images/Regression Plot", exist_ok=True)
    file_name = f"Images/Regression Plot/{model_name}_{dataset_type}_fit.png"
    plt.savefig(file_name)
    plt.close()

    return file_name

def log_residual_plot(y_true, y_pred, model_name):

    residuals = y_true - y_pred

    plt.figure()
    plt.scatter(y_pred, residuals, alpha=0.5)
    plt.axhline(y=0)

    plt.xlabel("Predicted Values")
    plt.ylabel("Residuals")
    plt.title(f"{model_name} - Residual Plot")

    os.makedirs("Images/Residual Plot", exist_ok=True)

    file_name = f"Images/Residual Plot/{model_name}_residuals.png"
    plt.savefig(file_name)
    plt.close()

    return file_name

# Training and Testing Data

In [ ]:
dataset_params = {
  'random_state':42,
  'test_size':0.2
}

X_train, X_test, y_train, y_test = train_test_split(
  X, y, random_state = dataset_params['random_state'], test_size=dataset_params['test_size']
)

# Tracking Model Experiments using MLFlow

In [ ]:
models = [
  (
    "Linear_Regression",
    lr_pipeline,
    (X_train, y_train),
    (X_test, y_test),
    lr_tags
  ),
  (
    "Ridge_Regression",
    ridge_grid,
    (X_train, y_train),
    (X_test, y_test),
    ridge_tags
  ),
  (
    "Lasso_Regression",
    lasso_grid,
    (X_train, y_train),
    (X_test, y_test),
    lasso_tags
  ),
  (
    "Decision_Tree_Regressor",
    dt_pipeline,
    (X_train, y_train),
    (X_test, y_test),
    dt_tags
  ),
  (
    "Random_Forest_Regressor",
    rf_pipeline,
    (X_train, y_train),
    (X_test, y_test),
    rf_tags
  ),
  (
    "XGBoost_Regressor",
    xgb_pipeline,
    (X_train, y_train),
    (X_test, y_test),
    xgb_tags
  )
]

In [ ]:
model_reports = []
model_tags = []

# -------------------------
# TRAINING LOOP
# -------------------------
for model_name, model, train_set, test_set, tags in models:

  model_tags.append(tags)

  X_train = train_set[0]
  y_train = train_set[1]
  X_test = test_set[0]
  y_test = test_set[1]

  model.fit(X_train, y_train)
  print(f"{model_name} Trained")

  y_train_pred = model.predict(X_train)
  y_test_pred = model.predict(X_test)

  model_metrics = {
    "train_r2": r2_score(y_train, y_train_pred),
    "test_r2": r2_score(y_test, y_test_pred),

    "train_mae": mean_absolute_error(y_train, y_train_pred),
    "test_mae": mean_absolute_error(y_test, y_test_pred),

    "train_rmse": np.sqrt(mean_squared_error(y_train, y_train_pred)),
    "test_rmse": np.sqrt(mean_squared_error(y_test, y_test_pred)),
  }

  model_reports.append(model_metrics)


# Dagshub Connection

In [ ]:
import dagshub

dagshub.init(
  repo_owner='JS-Tharun', 
  repo_name='Real-Estate-Investment-Advisor', 
  mlflow=True
)

In [ ]:
# -------------------------
# MLFLOW LOGGING LOOP
# -------------------------
load_dotenv()

os.environ['MLFLOW_TRACKING_USERNAME'] = f"{os.getenv('DAGSHUB_USERNAME')}"
os.environ['MLFLOW_TRACKING_PASSWORD'] = f"{os.getenv('DAGSHUB_PASSWORD')}"

mlflow.set_experiment(os.environ["MLFLOW_EXPERIMENT_NAME_REG"])
mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI'])

for i, element in enumerate(models):

  model_name = element[0]
  model = element[1]
  report = model_reports[i]

  X_train = element[2][0]
  y_train = element[2][1]
  X_test = element[3][0]
  y_test = element[3][1]

  dataset = from_pandas(X_train, name="X_train_dataset")
  model_tag = model_tags[i]

  with mlflow.start_run(run_name=model_name):

    mlflow.log_input(dataset, context='training_data')
    print("Input Logged")

    mlflow.set_tags(model_tag)
    print("Tag Logged")

    mlflow.log_param('model_name', model_name)

    # ✅ Handle GridSearchCV vs normal pipeline
    if hasattr(model, "best_estimator_"):
        model_to_log = model.best_estimator_

        mlflow.log_params(model.best_params_)
        mlflow.log_params(model_to_log.named_steps['model'].get_params())

        # ✅ Log best alpha explicitly
        if 'model__alpha' in model.best_params_:
            mlflow.log_param("best_alpha", model.best_params_['model__alpha'])

    else:
        model_to_log = model
        mlflow.log_params(model.named_steps['model'].get_params())

    print("Params Logged")

    mlflow.log_metrics(report)
    print("Metrics Logged")

    mlflow.sklearn.log_model(
      model_to_log, 
      model_name
    )
    print("Model Logged")

    # ✅🔥 CRITICAL FIX: Recompute predictions per model
    y_train_pred = model_to_log.predict(X_train)
    y_test_pred = model_to_log.predict(X_test)

    # Generate plots
    train_plot = log_regression_plot(y_train, y_train_pred, model_name, "train")
    test_plot = log_regression_plot(y_test, y_test_pred, model_name, "test")
    residual_plot = log_residual_plot(y_test, y_test_pred, model_name)

    # Log artifacts
    mlflow.log_artifact(train_plot, artifact_path="plots")
    mlflow.log_artifact(test_plot, artifact_path="plots")
    mlflow.log_artifact(residual_plot, artifact_path="plots")

    print("Plots Logged")

# Model Registration

In [ ]:
"""import dagshub

dagshub.init(
  repo_owner='JS-Tharun',
  repo_name='Real-Estate-Investment-Advisor',
  mlflow=True
)

mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI'])

model_name = "XGBoost_Regressor"
run_id = input("Enter model Run_ID")
model_uri = f"runs:/{run_id}/{model_name}"

result = mlflow.register_model(
  model_uri=model_uri, 
  name=model_name
)"""

# Load and Test the Model

In [ ]:
model_name = "XGBoost_Price_Pred"
model_uri=f"models:/{model_name}@challenger"

loaded_model = mlflow.pyfunc.load_model(model_uri)
y_pred = loaded_model.predict(X_test)
y_pred

# Transition the Model to Production

In [ ]:
dev_model_uri = f"models:/{model_name}@challenger"
prod_model = "Future_Price_Predictor-Prod"

client = mlflow.MlflowClient()

new_version = client.copy_model_version(src_model_uri=dev_model_uri, dst_name=prod_model)
client.set_registered_model_alias(
    name=prod_model,
    alias="champion",
    version=new_version.version
)

# Testing Production Model

In [ ]:
model_uri=f"models:/{prod_model}@champion"

loaded_model = mlflow.pyfunc.load_model(model_uri)
y_pred = loaded_model.predict(X_test)
y_pred